In [ ]:
import time
import bambulabs_api as bl
import os
import datetime

IP = '192.168.1.200'
SERIAL = 'AC12309BH109'
ACCESS_CODE = '12347890'

debug = os.getenv("debug", False)

if __name__ == '__main__':
    print('Starting bambulabs_api example')
    print('Connecting to Bambulabs 3D printer')
    print(f'IP: {IP}')
    print(f'Serial: {SERIAL}')
    print(f'Access Code: {ACCESS_CODE}')

    # Create a new instance of the API
    printer = bl.Printer(IP, ACCESS_CODE, SERIAL)

    # Connect to the Bambulabs 3D printer
    printer.connect()

    try:
        while True:
            time.sleep(5)

            # Get the printer status
            status = printer.get_state()
            percentage = printer.get_percentage()
            layer_num = printer.current_layer_num()
            total_layer_num = printer.total_layer_num()
            bed_temperature = printer.get_bed_temperature()
            nozzle_temperature = printer.get_nozzle_temperature()
            remaining_time = printer.get_time()
            if remaining_time is not None:
                finish_time = datetime.datetime.now() + datetime.timedelta(
                    minutes=int(remaining_time))
                finish_time_format = finish_time.strftime("%Y-%m-%d %H:%M:%S")
            else:
                finish_time_format = "NA"

            print(
                f'''Printer status: {status}
                Layers: {layer_num}/{total_layer_num}
                percentage: {percentage}%
                Bed temp: {bed_temperature} ºC
                Nozzle temp: {nozzle_temperature} ºC
                Remaining time: {remaining_time}m
                Finish time: {finish_time_format}
                '''
            )

            if debug:
                import json
                print("=" * 100)
                print("Printer MQTT Dump")
                print(json.dumps(
                    printer.mqtt_dump(),
                    sort_keys=True,
                    indent=2
                ))
                print("=" * 100)
    finally:
        # Disconnect from the Bambulabs 3D printer
        printer.disconnect()

In [ ]:
from io import BytesIO
import time
import zipfile
import bambulabs_api as bl
import os

IP = '172.16.97.160'
SERIAL = '01P00C641402808'
ACCESS_CODE = '42117904'

# ============================================================
INPUT_FILE_PATH = 'modified.gcode'
UPLOAD_FILE_NAME = 'modified.3mf'
# ============================================================

env = os.getenv("env", "debug")
plate = os.getenv("plate", "true").lower() == "true"


def create_zip_archive_in_memory(
        text_content: str,
        text_file_name: str = 'file.txt') -> BytesIO:
    """
    Create a zip archive in memory

    Args:
        text_content (str): content of the text file
        text_file_name (str, optional): location of the text file.
            Defaults to 'file.txt'.

    Returns:
        io.BytesIO: zip archive in memory
    """
    zip_buffer = BytesIO()
    with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.writestr(text_file_name, text_content)
    zip_buffer.seek(0)
    return zip_buffer


if __name__ == '__main__':
    print('Starting bambulabs_api example')
    print('Connecting to Bambulabs 3D printer')
    print(f'IP: {IP}')
    print(f'Serial: {SERIAL}')
    print(f'Access Code: {ACCESS_CODE}')

    # Create a new instance of the API
    printer = bl.Printer(IP, ACCESS_CODE, SERIAL)

    # Connect to the Bambulabs 3D printer
    printer.connect()

    time.sleep(5)

    with open(INPUT_FILE_PATH, "r") as file:
        gcode = file.read()

    if plate:
        gcode_location = "Metadata/plate_1.gcode"
        io_file = create_zip_archive_in_memory(gcode, gcode_location)
        if file:
            result = printer.upload_file(io_file, UPLOAD_FILE_NAME)
            if "226" not in result:
                print("Error Uploading File to Printer")

            else:
                print("Done Uploading/Sending Start Print Command")
                #printer.start_print(UPLOAD_FILE_NAME, 1)
                #print("Start Print Command Sent")
    else:
        gcode_location = INPUT_FILE_PATH
        io_file = create_zip_archive_in_memory(gcode, gcode_location)
        if file:
            result = printer.upload_file(io_file, UPLOAD_FILE_NAME)
            if "226" not in result:
                print("Error Uploading File to Printer")

            else:
                print("Done Uploading/Sending Start Print Command")
                #printer.start_print(UPLOAD_FILE_NAME, gcode_location)
                #print("Start Print Command Sent")